In [1]:
import numpy as np
import pandas as pd

# sklearn models
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# xgboost
from xgboost import XGBClassifier

c:\Users\HP\anaconda3\envs\ascend_env\lib\site-packages\xgboost\compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
c:\Users\HP\anaconda3\envs\ascend_env\lib\site-packages\xgboost\compat.py:106: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [6]:
# =========================================================
# 0. IMPORT
# =========================================================
import pandas as pd

# =========================================================
# 1. LOAD DATA
# =========================================================
df_customer = pd.read_csv("customer_candidate_selection.csv")
df_transaction = pd.read_csv("transaction_candidate_selection.csv")

print("✅ Data Loaded")

# =========================================================
# 2. BASIC CHECK
# =========================================================
print("\n=== CUSTOMER DATA ===")
print(df_customer.head())

print("\n=== TRANSACTION DATA ===")
print(df_transaction.head())

# =========================================================
# 3. CONVERT DATE TYPES
# =========================================================
df_customer['RegistrationDate'] = pd.to_datetime(df_customer['RegistrationDate'], errors='coerce')
df_transaction['Timestamp'] = pd.to_datetime(df_transaction['Timestamp'], errors='coerce')

print("\n✅ Date conversion done")

✅ Data Loaded

=== CUSTOMER DATA ===
   CustomerID   Age        Gender Location RegistrationDate  WalletBalance  \
0     3074082  65.0  Unidentified  BANGKOK       2023-05-27         158.56   
1     5545792  56.0  Unidentified      NaN       2023-11-03         677.50   
2     6756636  67.0  Unidentified      NaN       2024-01-13           6.40   
3     6835172  61.0  Unidentified  BANGKOK       2024-01-18         346.12   
4     4711710  59.0  Unidentified  BANGKOK       2023-09-06          31.38   

  LoyaltyTier  
0        Gold  
1         NaN  
2    Platinum  
3        Gold  
4         NaN  

=== TRANSACTION DATA ===
   TransactionID  CustomerID                Timestamp TransactionType  Amount  \
0           3836     1707873  2024-01-05 23:07:16 UTC         Payment     2.0   
1         549913      539132  2024-02-02 16:41:41 UTC         Payment     2.0   
2         919618      138973  2024-03-27 21:37:08 UTC        Transfer     2.0   
3         563684     1197450  2024-02-04 18:29:1

In [12]:
# =========================================================
# 4. DATA QUALITY CHECK
# =========================================================
#-- Customer data missing value check
missing_df = (
    df_customer.isna().sum()
    .to_frame(name='missing_count')
)

missing_df['missing_percent'] = (missing_df['missing_count'] / len(df_customer)) * 100

missing_df = missing_df.sort_values(by='missing_percent', ascending=False)

print("\n=== MISSING VALUES (CUSTOMER) ===")
print(missing_df)

#-- Transaction data missing value check
missing_tx = (
    df_transaction.isna().sum()
    .to_frame(name='missing_count')
)

missing_tx['missing_percent'] = (missing_tx['missing_count'] / len(df_transaction)) * 100

missing_tx = missing_tx.sort_values(by='missing_percent', ascending=False)

print("\n=== MISSING VALUES (TRANSACTION) ===")
print(missing_tx)


=== MISSING VALUES (CUSTOMER) ===
                  missing_count  missing_percent
Location                 184130        60.862983
Age                      142463        47.090225
LoyaltyTier               50412        16.663361
CustomerID                    0         0.000000
Gender                        0         0.000000
RegistrationDate              0         0.000000
WalletBalance                 0         0.000000

=== MISSING VALUES (TRANSACTION) ===
                 missing_count  missing_percent
Location                107818         8.315767
PaymentMethod            69015         5.322977
TransactionID                0         0.000000
CustomerID                   0         0.000000
Timestamp                    0         0.000000
TransactionType              0         0.000000
Amount                       0         0.000000
Category                     0         0.000000


In [13]:
# =========================================================
# 5. LOYALTY TIER ANALYSIS
# =========================================================
print("\n=== LOYALTY TIER UNIQUE ===")
print(df_customer['LoyaltyTier'].unique())

print("\n=== LOYALTY TIER DISTRIBUTION ===")
print(df_customer['LoyaltyTier'].value_counts())

print("\n=== LOYALTY TIER % ===")
print(df_customer['LoyaltyTier'].value_counts(normalize=True))


=== LOYALTY TIER UNIQUE ===
['Gold' nan 'Platinum' 'Silver']

=== LOYALTY TIER DISTRIBUTION ===
Gold        100962
Platinum    100912
Silver       50246
Name: LoyaltyTier, dtype: int64

=== LOYALTY TIER % ===
Gold        0.400452
Platinum    0.400254
Silver      0.199294
Name: LoyaltyTier, dtype: float64


In [14]:
# =========================================================
# 6. BASIC INSIGHT CHECK
# =========================================================
print("\n=== AVG WALLET BY TIER ===")
print(df_customer.groupby('LoyaltyTier')['WalletBalance'].mean())

print("\n=== COUNT CUSTOMER BY TIER ===")
print(df_customer.groupby('LoyaltyTier')['CustomerID'].count())


=== AVG WALLET BY TIER ===
LoyaltyTier
Gold        499.707703
Platinum    500.392375
Silver      499.933984
Name: WalletBalance, dtype: float64

=== COUNT CUSTOMER BY TIER ===
LoyaltyTier
Gold        100962
Platinum    100912
Silver       50246
Name: CustomerID, dtype: int64


In [17]:
# =========================================================
# 7. CLEAN LOYALTY TIER (OPTIONAL BUT RECOMMENDED)
# =========================================================
df_customer['LoyaltyTier'] = df_customer['LoyaltyTier'].str.strip().str.title()

print("\n=== CLEANED LOYALTY TIER ===")
print(df_customer['LoyaltyTier'].value_counts())

print("\n✅ EDA DONE")
print(len(df_customer))


=== CLEANED LOYALTY TIER ===
Gold        100962
Platinum    100912
Silver       50246
Name: LoyaltyTier, dtype: int64

✅ EDA DONE
302532


In [22]:
print(len(df_customer))  

302532


In [48]:
for col in df_customer.columns:
    unique_customer = df_customer[col].unique()
    print(f"\n=== {col} ===")
    print(f"Unique count: {len(unique_customer)}")
    print(unique_customer)

# =========================================================
# 1. DISTRIBUTION
# =========================================================
gender_dist = df_customer['Gender'].value_counts(dropna=False)

print("\n=== GENDER DISTRIBUTION ===")
print(gender_dist)

# =========================================================
# 2. % ของแต่ละกลุ่ม
# =========================================================
gender_percent = df_customer['Gender'].value_counts(normalize=True, dropna=False) * 100

print("\n=== GENDER % ===")
print(gender_percent)

df_2024 = df_customer[df_customer['year'] == 2024].copy()
gender_2024_dist = df_2024['Gender'].value_counts(dropna=False)

print("\n=== GENDER DISTRIBUTION (2024) ===")
print(gender_2024_dist)
gender_2024_percent = df_2024['Gender'].value_counts(normalize=True, dropna=False) * 100

print("\n=== GENDER % (2024) ===")
print(gender_2024_percent)


=== CustomerID ===
Unique count: 302532
[3074082 5545792 6756636 ... 2919596 7504393 4310559]

=== Age ===
Unique count: 89
[65. 56. 67. 61. 59. 60. 79. 14. 64. 58. 13. 57. 63. 74. 76. 11. 66. 62.
 72. 86. 68. 70. 71. 77. 69. 73. 75. 12. 84. 83. 81. 78. 93. 92. 80. 82.
 88. 91. 87. 10. 85. 89. 90. 99. nan 96. 94. 95. 15. 16. 17. 18. 19. 20.
 21. 22. 23. 24. 25. 26. 27. 28. 29. 30. 31. 32. 33. 34. 35. 36. 37. 38.
 39. 40. 41. 42. 43. 44. 45. 46. 47. 48. 49. 50. 51. 52. 53. 54. 55.]

=== Gender ===
Unique count: 3
['Unidentified' 'Female' 'Male']

=== Location ===
Unique count: 77
<StringArray>
[                 'BANGKOK',                       <NA>,
              'SUPHAN BURI',               'CHIANG MAI',
             'CHACHOENGSAO',              'PHITSANULOK',
                   'RAYONG',                 'CHONBURI',
                  'LAMPANG',                   'PHUKET',
                  'PATTANI',            'NAKHON PATHOM',
                  'LOPBURI',        'NAKHON RATCHASIMA',


In [45]:
for col in df_transaction.columns:
    unique_transaction = df_transaction[col].unique()
    print(f"\n=== {col} ===")
    print(f"Unique count: {len(unique_transaction)}")
    print(unique_transaction)


=== TransactionID ===
Unique count: 1296549
[   3836  549913  919618 ...  228399  447171 1565280]

=== CustomerID ===
Unique count: 60180
[1707873  539132  138973 ... 2291318 1222066 1230834]

=== Timestamp ===
Unique count: 1220529
<DatetimeArray>
['2024-01-05 23:07:16+00:00', '2024-02-02 16:41:41+00:00',
 '2024-03-27 21:37:08+00:00', '2024-02-04 18:29:11+00:00',
 '2024-02-02 17:06:47+00:00', '2024-05-14 17:32:06+00:00',
 '2024-01-09 12:15:45+00:00', '2024-04-06 14:06:42+00:00',
 '2024-04-28 18:16:20+00:00', '2024-04-16 19:29:26+00:00',
 ...
 '2024-02-07 22:56:35+00:00', '2024-03-24 17:53:22+00:00',
 '2024-01-22 06:49:27+00:00', '2024-02-07 16:04:28+00:00',
 '2024-04-08 15:08:58+00:00', '2024-04-22 05:15:32+00:00',
 '2024-04-06 16:14:51+00:00', '2024-02-24 15:33:37+00:00',
 '2024-03-06 21:03:37+00:00', '2024-01-16 20:22:55+00:00']
Length: 1220529, dtype: datetime64[ns, UTC]

=== TransactionType ===
Unique count: 3
['Payment' 'Transfer' 'Bill Payment']

=== Amount ===
Unique count: 50

In [27]:
# =========================================================
# 3. CREATE YEAR COLUMN
# =========================================================
df_customer['year'] = df_customer['RegistrationDate'].dt.year
df_transaction['year'] = df_transaction['Timestamp'].dt.year

# =========================================================
# 4. BASIC CLEAN (IMPORTANT)
# =========================================================

# Clean Age (invalid → NaN)
df_customer.loc[(df_customer['Age'] <= 0) | (df_customer['Age'] > 100), 'Age'] = pd.NA

# Normalize string
df_customer['Location'] = df_customer['Location'].astype('string').str.strip().str.upper()
df_transaction['Location'] = df_transaction['Location'].astype('string').str.strip().str.upper()
df_transaction['PaymentMethod'] = df_transaction['PaymentMethod'].astype('string').str.strip().str.upper()

# =========================================================
# 5. FUNCTION: MISSING % BY YEAR (CORRECT LOGIC)
# =========================================================
def missing_percent_by_year(df, col):
    return (
        df.groupby('year')[col]
        .apply(lambda x: x.isna().mean() * 100)
        .reset_index(name='missing_percent')
        .sort_values('year')
    )

# =========================================================
# 6. CUSTOMER ANALYSIS
# =========================================================

print("\n=== CUSTOMER: AGE MISSING (%) ===")
missing_age = missing_percent_by_year(df_customer, 'Age')
print(missing_age)

print("\n=== CUSTOMER: LOCATION MISSING (%) ===")
missing_location_customer = missing_percent_by_year(df_customer, 'Location')
print(missing_location_customer)

# =========================================================
# 7. TRANSACTION ANALYSIS
# =========================================================

print("\n=== TRANSACTION: PAYMENT METHOD MISSING (%) ===")
missing_payment = missing_percent_by_year(df_transaction, 'PaymentMethod')
print(missing_payment)

print("\n=== TRANSACTION: LOCATION MISSING (%) ===")
missing_location_tx = missing_percent_by_year(df_transaction, 'Location')
print(missing_location_tx)


=== CUSTOMER: AGE MISSING (%) ===
   year  missing_percent
0  2016        10.000000
1  2017         0.000000
2  2018         2.302632
3  2019         2.839397
4  2020         9.423523
5  2021        16.889752
6  2022        23.350607
7  2023        28.122683
8  2024        73.807039

=== CUSTOMER: LOCATION MISSING (%) ===
   year  missing_percent
0  2016        10.000000
1  2017         0.000000
2  2018         1.973684
3  2019         1.952085
4  2020         8.959486
5  2021        16.164300
6  2022        29.781240
7  2023        51.349788
8  2024        82.314664

=== TRANSACTION: PAYMENT METHOD MISSING (%) ===
   year  missing_percent
0  2024         5.322977

=== TRANSACTION: LOCATION MISSING (%) ===
   year  missing_percent
0  2024         8.315767


In [29]:
df_2024 = df_customer[df_customer['year'] == 2024].copy()

df_2024['month'] = df_2024['RegistrationDate'].dt.month

missing_age_2024_month = (
    df_2024
    .groupby('month')['Age']
    .apply(lambda x: x.isna().mean() * 100)
    .reset_index(name='missing_percent')
    .sort_values('month')
)

missing_age_2024_detail = (
    df_2024
    .groupby('month')['Age']
    .agg(
        total='size',
        missing=lambda x: x.isna().sum()
    )
    .reset_index()
)

missing_age_2024_detail['missing_percent'] = (
    missing_age_2024_detail['missing'] / missing_age_2024_detail['total'] * 100
)

print(missing_age_2024_detail)


    month  total  missing  missing_percent
0       1  12248     4122        33.654474
1       2  10368     3537        34.114583
2       3  21656    16244        75.009235
3       4  27864    23030        82.651450
4       5  31886    27385        85.884087
5       6  24788    21316        85.993223
6       7   1827     1355        74.165298
7       8   1569      663        42.256214
8       9    858      625        72.843823
9      10    581      418        71.944923
10     11    387      274        70.801034
11     12    130       52        40.000000


In [31]:
df_tx_2024 = df_transaction[df_transaction['year'] == 2024].copy()

df_tx_2024['month'] = df_tx_2024['Timestamp'].dt.month

missing_location_month = (
    df_tx_2024
    .groupby('month')['Location']
    .apply(lambda x: x.isna().mean() * 100)
    .reset_index(name='missing_percent')
    .sort_values('missing_percent', ascending=False)
)

print(missing_location_month)

missing_location_detail = (
    df_tx_2024
    .groupby('month')['Location']
    .agg(
        total='size',
        missing=lambda x: x.isna().sum()
    )
    .reset_index()
)

missing_location_detail['missing_percent'] = (
    missing_location_detail['missing'] / missing_location_detail['total'] * 100
)

missing_location_detail = missing_location_detail.sort_values(
    'missing_percent', ascending=False
)

print(missing_location_detail)

   month  missing_percent
5      6        10.608182
4      5         9.213676
0      1         8.644446
1      2         8.247313
2      3         7.713854
3      4         7.345868
   month   total  missing  missing_percent
5      6   12858     1364        10.608182
4      5  365066    33636         9.213676
0      1  198347    17146         8.644446
1      2  206637    17042         8.247313
2      3  244197    18837         7.713854
3      4  269444    19793         7.345868


In [32]:
df_transaction['Timestamp'] = pd.to_datetime(df_transaction['Timestamp'], errors='coerce')

start_date = df_transaction['Timestamp'].min()
end_date = df_transaction['Timestamp'].max()

print("Start Date:", start_date)
print("End Date:", end_date)

Start Date: 2024-01-01 00:01:08+00:00
End Date: 2024-06-01 23:59:45+00:00


In [33]:
df_missing = df_transaction[df_transaction['Location'].isna()]

total_missing_rows = len(df_missing)
unique_customers = df_missing['CustomerID'].nunique()

print("Total missing rows:", total_missing_rows)
print("Unique customers:", unique_customers)

ratio = unique_customers / total_missing_rows

print("Customer uniqueness ratio:", ratio)

top_missing_customers = (
    df_missing['CustomerID']
    .value_counts()
    .head(10)
)

print(top_missing_customers)

missing_per_customer = df_missing['CustomerID'].value_counts()

print(missing_per_customer.describe())

customer_total = df_transaction.groupby('CustomerID').size()
customer_missing = df_missing.groupby('CustomerID').size()

customer_missing_ratio = (customer_missing / customer_total).fillna(0)

print(customer_missing_ratio.describe())

Total missing rows: 107818
Unique customers: 15637
Customer uniqueness ratio: 0.1450314418742696
1345553    267
1771115    228
1747365    209
1455602    166
487330     164
1610051    161
1751585    155
1758532    137
1001858    137
1071927    134
Name: CustomerID, dtype: int64
count    15637.000000
mean         6.895057
std         10.931684
min          1.000000
25%          1.000000
50%          3.000000
75%          8.000000
max        267.000000
Name: CustomerID, dtype: float64
count    60180.000000
mean         0.258327
std          0.436962
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000
dtype: float64


In [34]:
# ลูกค้าทั้งหมด
total_customers = df_transaction['CustomerID'].nunique()

# ลูกค้าที่มี missing อย่างน้อย 1 ครั้ง
missing_customers = df_transaction[
    df_transaction['Location'].isna()
]['CustomerID'].nunique()

# %
missing_customer_percent = (missing_customers / total_customers) * 100

print("Total customers:", total_customers)
print("Customers with missing Location:", missing_customers)
print(f"Missing customer %: {missing_customer_percent:.2f}%")

customer_total_tx = df_transaction.groupby('CustomerID').size()
customer_missing_tx = df_transaction[df_transaction['Location'].isna()] \
    .groupby('CustomerID').size()

# % missing ต่อ customer
customer_missing_ratio = (customer_missing_tx / customer_total_tx).fillna(0)

# ลูกค้าที่ missing 100%
fully_missing_customers = (customer_missing_ratio == 1).sum()

percent_fully_missing = fully_missing_customers / total_customers * 100

print("Customers with 100% missing:", fully_missing_customers)
print(f"% Fully missing customers: {percent_fully_missing:.2f}%")

Total customers: 60180
Customers with missing Location: 15637
Missing customer %: 25.98%
Customers with 100% missing: 15433
% Fully missing customers: 25.64%


In [35]:
# =========================================================
# 1. PREP
# =========================================================
import pandas as pd

df_transaction['Timestamp'] = pd.to_datetime(df_transaction['Timestamp'], errors='coerce')

# =========================================================
# 2. AGGREGATE PER CUSTOMER
# =========================================================
customer_stats = (
    df_transaction
    .groupby('CustomerID')
    .agg(
        total_tx=('TransactionID', 'count'),
        missing_location=('Location', lambda x: x.isna().sum())
    )
    .reset_index()
)

# =========================================================
# 3. CALCULATE % MISSING
# =========================================================
customer_stats['missing_percent'] = (
    customer_stats['missing_location'] / customer_stats['total_tx'] * 100
)

# =========================================================
# 4. DEFINE FREQUENCY SEGMENT (ซื้อบ่อยไหม)
# =========================================================
def classify_customer(tx):
    if tx >= 50:
        return 'High Frequency'
    elif tx >= 10:
        return 'Medium Frequency'
    else:
        return 'Low Frequency'

customer_stats['frequency_segment'] = customer_stats['total_tx'].apply(classify_customer)

# =========================================================
# 5. SUMMARY BY SEGMENT
# =========================================================
segment_summary = (
    customer_stats
    .groupby('frequency_segment')
    .agg(
        avg_tx=('total_tx', 'mean'),
        avg_missing_percent=('missing_percent', 'mean'),
        customer_count=('CustomerID', 'count')
    )
    .reset_index()
)

print("\n=== SEGMENT SUMMARY ===")
print(segment_summary)

# =========================================================
# 6. CHECK HIGH VALUE CUSTOMERS (top 10%)
# =========================================================
threshold = customer_stats['total_tx'].quantile(0.9)

high_value_customers = customer_stats[customer_stats['total_tx'] >= threshold]

print("\n=== HIGH VALUE CUSTOMER MISSING ===")
print(high_value_customers['missing_percent'].describe())

# =========================================================
# 7. OVERALL IMPACT
# =========================================================
total_tx_all = customer_stats['total_tx'].sum()
total_missing_all = customer_stats['missing_location'].sum()

overall_missing_percent = total_missing_all / total_tx_all * 100

print("\n=== OVERALL ===")
print("Total Transactions:", total_tx_all)
print("Total Missing:", total_missing_all)
print(f"Overall Missing %: {overall_missing_percent:.2f}%")

# =========================================================
# 8. OPTIONAL: SORT TOP BAD CUSTOMERS
# =========================================================
print("\n=== TOP CUSTOMERS WITH HIGH MISSING ===")
print(customer_stats.sort_values('missing_percent', ascending=False).head(10))


=== SEGMENT SUMMARY ===
  frequency_segment      avg_tx  avg_missing_percent  customer_count
0    High Frequency  176.757355             3.716623            4521
1     Low Frequency    3.570799            32.213547           38327
2  Medium Frequency   20.803773            17.491542           17332

=== HIGH VALUE CUSTOMER MISSING ===
count    6095.000000
mean        5.258796
std        22.240303
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       100.000000
Name: missing_percent, dtype: float64

=== OVERALL ===
Total Transactions: 1296549
Total Missing: 107818
Overall Missing %: 8.32%

=== TOP CUSTOMERS WITH HIGH MISSING ===
       CustomerID  total_tx  missing_location  missing_percent  \
33848     1619522         1                 1            100.0   
13192      629097         3                 3            100.0   
13199      629444        11                11            100.0   
13198      629345         4                 4            10

In [43]:
top_customers_by_tx = customer_stats.sort_values(
    by='total_tx', ascending=False
)

print("\n=== TOP CUSTOMERS BY TOTAL_TX ===")
print(top_customers_by_tx.head(20))

top_customers_with_missing = customer_stats.sort_values(
    by='total_tx', ascending=False
)[['CustomerID', 'total_tx', 'missing_location', 'missing_percent']]

print("\n=== TOP CUSTOMERS + MISSING ===")
print(top_customers_with_missing.head(20))

top_missing_location = customer_stats.sort_values(
    by='missing_location', ascending=False
)

print("\n=== TOP CUSTOMERS (MISSING LOCATION สูงสุด) ===")
print(top_missing_location[['CustomerID', 'missing_location', 'total_tx', 'missing_percent']].head(20))

# =========================================================
# 1. TOTAL TRANSACTION ทั้ง dataset
# =========================================================
total_tx_all = df_transaction.shape[0]

# =========================================================
# 2. เพิ่ม column impact ต่อ dataset
# =========================================================
customer_stats['missing_tx_ratio_global'] = (
    customer_stats['missing_location'] / total_tx_all * 100
)

# =========================================================
# 3. เรียงใหม่ + แสดงผล
# =========================================================
top_missing_location = customer_stats.sort_values(
    by='missing_location', ascending=False
)

print("\n=== TOP CUSTOMERS (WITH GLOBAL IMPACT) ===")
print(top_missing_location[
    ['CustomerID', 'missing_location', 'total_tx', 'missing_percent', 'missing_tx_ratio_global']
].head(20))

# =========================================================
# 1. TOTAL TRANSACTION
# =========================================================
total_tx_all = df_transaction.shape[0]

# =========================================================
# 2. GLOBAL % (ต่อ dataset ทั้งหมด)
# =========================================================
customer_stats['missing_tx_percent_global'] = (
    customer_stats['missing_location'] / total_tx_all * 100
)

# =========================================================
# 3. ROUND ให้สวย
# =========================================================
customer_stats['missing_tx_percent_global'] = customer_stats[
    'missing_tx_percent_global'
].round(4)

# =========================================================
# 4. SORT + SHOW
# =========================================================
top_missing_location = customer_stats.sort_values(
    by='missing_location', ascending=False
)

print("\n=== TOP CUSTOMERS (GLOBAL % IMPACT) ===")
print(top_missing_location[
    ['CustomerID', 'missing_location', 'total_tx', 'missing_percent', 'missing_tx_percent_global']
].head(20))


=== TOP CUSTOMERS BY TOTAL_TX ===
       CustomerID  total_tx  missing_location  missing_percent  \
6389       305538      7510                 0              0.0   
28123     1341639      6109                 0              0.0   
13550      645362      4544                 0              0.0   
39309     1887110      3517                 0              0.0   
6077       289088      2913                 0              0.0   
11770      561813      2890                 0              0.0   
2460       118741      2806                 0              0.0   
20167      956708      2746                 0              0.0   
9098       434591      2613                 0              0.0   
8271       395517      2552                 0              0.0   
25299     1205789      2450                 0              0.0   
26737     1272893      2372                 0              0.0   
3390       162642      2111                 0              0.0   
36891     1770873      2084              

In [41]:
# =========================================================
# 1. TOTAL CUSTOMER (มี transaction)
# =========================================================
total_customers = df_transaction['CustomerID'].nunique()

# =========================================================
# 2. CUSTOMER ที่มี missing Location อย่างน้อย 1 ครั้ง
# =========================================================
customers_with_missing = df_transaction[
    df_transaction['Location'].isna()
]['CustomerID'].nunique()

percent_customers_missing = customers_with_missing / total_customers * 100

print("\n=== CUSTOMER IMPACT (ANY MISSING) ===")
print("Total customers:", total_customers)
print("Customers with missing:", customers_with_missing)
print(f"% customers affected: {percent_customers_missing:.2f}%")

# =========================================================
# 3. CUSTOMER ที่ missing เยอะ (เช่น >50%)
# =========================================================
customer_total_tx = df_transaction.groupby('CustomerID').size()
customer_missing_tx = df_transaction[df_transaction['Location'].isna()] \
    .groupby('CustomerID').size()

customer_missing_ratio = (customer_missing_tx / customer_total_tx).fillna(0)

# threshold = 50%
heavy_missing_customers = (customer_missing_ratio > 0.5).sum()

percent_heavy_missing = heavy_missing_customers / total_customers * 100

print("\n=== CUSTOMER IMPACT (HEAVY MISSING >50%) ===")
print("Customers with heavy missing:", heavy_missing_customers)
print(f"% heavy missing customers: {percent_heavy_missing:.2f}%")

# =========================================================
# 4. CUSTOMER ที่ missing 100% (หนักสุด)
# =========================================================
fully_missing_customers = (customer_missing_ratio == 1).sum()

percent_fully_missing = fully_missing_customers / total_customers * 100

print("\n=== CUSTOMER IMPACT (FULLY MISSING) ===")
print("Customers fully missing:", fully_missing_customers)
print(f"% fully missing customers: {percent_fully_missing:.2f}%")


=== CUSTOMER IMPACT (ANY MISSING) ===
Total customers: 60180
Customers with missing: 15637
% customers affected: 25.98%

=== CUSTOMER IMPACT (HEAVY MISSING >50%) ===
Customers with heavy missing: 15531
% heavy missing customers: 25.81%

=== CUSTOMER IMPACT (FULLY MISSING) ===
Customers fully missing: 15433
% fully missing customers: 25.64%


In [44]:
partial_missing_customers = customer_stats[
    (customer_stats['missing_location'] >= 1) &
    (customer_stats['total_tx'] != customer_stats['missing_location'])
]

print("\n=== CUSTOMERS WITH PARTIAL MISSING ===")
print(partial_missing_customers.head(20))


=== CUSTOMERS WITH PARTIAL MISSING ===
       CustomerID  total_tx  missing_location  missing_percent  \
3513       168526         2                 1        50.000000   
5280       249585        30                13        43.333333   
7302       349886        14                10        71.428571   
7765       370798        11                 8        72.727273   
8038       384341        24                23        95.833333   
9653       460224         6                 2        33.333333   
10612      504632        15                 3        20.000000   
14622      694352        32                 6        18.750000   
14801      702312        59                25        42.372881   
15796      750758         4                 3        75.000000   
16307      776275        10                 3        30.000000   
16664      792147        17                10        58.823529   
17086      814127         6                 1        16.666667   
17575      838254         7         